# ADO · Real Waymo trajectory learning
Train an original three-camera trajectory model on the Waymo End-to-End Driving dataset. The driver predicts 20 ground-plane waypoints over five seconds. This notebook runs on your Colab GPU and keeps raw records and feature arrays private. Use a Google account with [Waymo dataset access](https://waymo.com/open/download/) and review [Waymo’s non-commercial terms](https://waymo.com/open/terms/). The official Waymo challenge has a different evaluation protocol.

In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/devstorage.read_only'])
access = AuthorizedSession(credentials).get('https://storage.googleapis.com/storage/v1/b/waymo_open_dataset_end_to_end_camera_v_1_0_0/o', params={'maxResults':1, 'fields':'items(name)'}, timeout=20)
access.raise_for_status()
print('Official Waymo storage access verified.')

In [ ]:
from pathlib import Path
import subprocess, os, sys
repo = Path('/content/ado-waymo')
if not (repo / 'waymo_training/colab_job.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', '--branch', 'research/waymo-e2e', 'https://github.com/yafi-s/ADO.git', str(repo)], check=True)
    subprocess.run(['git','sparse-checkout','set','waymo_training'],cwd=repo,check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','waymo_training/requirements-colab.txt'],check=True)
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],check=True)

## Extract, train, evaluate
The default pilot reads at most 4.75 GiB of shard prefixes before retries, with a hard combined transfer budget of 5 GiB. It decodes only the three chosen views into compact features and fits vision, history and constant-velocity comparisons on the same examples. The command resumes saved shards and completed epochs. To make them survive runtime deletion, optionally mount your private Google Drive and change the two paths below.

In [ ]:
DATA = '/content/ado-data'
RUN = '/content/ado-run'
subprocess.run([sys.executable,'-u','-m','waymo_training.colab_job','--data',DATA,'--output',RUN],check=True)

In [ ]:
import json
report = json.loads((Path(RUN) / 'evaluation.json').read_text())
print(json.dumps(report['splits']['test']['seed_mean'], indent=2))

Keep the trained checkpoints and dataset features private unless you meet the Waymo agreement’s conditions for distribution. The [research notice](https://github.com/yafi-s/ADO/blob/research/waymo-e2e/waymo_training/NOTICE.md) explains the attribution and terms. Release the Colab runtime when finished.